In [1]:
# 导入训练和测试数据集
import pandas as pd
from datasets import Dataset
import re

df = pd.read_csv('train.csv')

drop_df = df.drop(['keyword','location'],axis=1)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\W+', ' ', text)

    return text 

drop_df['text'] = drop_df['text'].apply(clean_text)
dataset = Dataset.from_pandas(drop_df)

D:\Anaconda\envs\ml-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from datasets import load_metric
from transformers import BertTokenizerFast,BertForSequenceClassification
from transformers import TrainingArguments,Trainer

model_path = "bert-base-uncased"
tokenizer = BertTokenizerFast.from_pretrained(model_path)

#分词
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=27)
    
#map可在样本上批量执行操作
tokenized_dataset = dataset.map(tokenize_function, batched=True)

def transform_dataset(tokenized_dataset):
    tokenized_dataset = tokenized_dataset.remove_columns(["id"])
    tokenized_dataset = tokenized_dataset.remove_columns(["text"])
    tokenized_dataset = tokenized_dataset.rename_column("target", "labels")
   
    return tokenized_dataset

new_dataset = transform_dataset(tokenized_dataset)
split_dataset = new_dataset.train_test_split(test_size=0.1)

train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

Map: 100%|██████████| 7613/7613 [00:00<00:00, 22618.69 examples/s]


In [3]:
#基于Transformers提供的Trainer进行训练和测试
import numpy as np
model = BertForSequenceClassification.from_pretrained(model_path, num_labels=2)

import evaluate
metric = evaluate.load('accuracy')
# 设置每轮的验证过程
def compute_metrics(eval_pred):
    x,y = eval_pred
    pred = np.argmax(x, axis=1)
    return metric.compute(predictions=pred, references=y)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
# 设置训练参数
training_args = TrainingArguments(
    output_dir="model_output",# 指定模型输出和检查点的保存目录
    eval_strategy="epoch",# 每个训练时期结束后进行评估
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    load_best_model_at_end=True,
    num_train_epochs=6,
    weight_decay=0.1,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.385492,0.850394
2,0.018200,1.421495,0.826772
3,0.016900,1.360928,0.821522
4,0.014200,1.311982,0.850394
5,0.010600,1.422856,0.841207
6,0.009400,1.419437,0.839895


TrainOutput(global_step=2574, training_loss=0.013587663547883409, metrics={'train_runtime': 322.659, 'train_samples_per_second': 127.398, 'train_steps_per_second': 7.977, 'total_flos': 570345629148360.0, 'train_loss': 0.013587663547883409, 'epoch': 6.0})

In [14]:
import torch

test = pd.read_csv('test.csv')
test['text'] = test['text'].apply(clean_text)
test_dataset = test['text'].tolist()
tokenized_test_dataset = tokenizer(test_dataset, padding="max_length", truncation=True, max_length=27, return_tensors="pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenized_test_data = {key: value.to(device) for key, value in tokenized_test_dataset.items()}

# 使用模型进行预测
with torch.no_grad():
    logits = model(**tokenized_test_data).logits
    logits_cpu = logits.cpu().numpy()
    pred = np.argmax(logits_cpu, axis=1)

In [15]:
# 将预测结果添加到 submission.csv
df = pd.DataFrame(pred, columns=['target'])
submission = pd.DataFrame({'id': test['id'], 'target': pred})

# 保存结果
submission.to_csv('submission.csv', index=False)